In [1]:
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset

2026-02-18 14:15:30.645597: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
filepath = "ProcessedData/WebApplicationsCorpus.xlsx"
df = pd.read_excel(filepath)
df.head(5)

,data,intent
0,Alternative to Facebook,Find Alternative
1,How do I delete my Facebook account?,Delete Account
2,Are there any good Pandora alternatives with g...,Find Alternative
3,Is it possible to export my data from Trello t...,Export Data
4,Is there an online alternative to iGoogle,Find Alternative


In [3]:
df.intent.value_counts()

intent
Find Alternative    23
Filter Spam         20
Delete Account      17
Sync Accounts        9
Change Password      8
Export Data          5
Name: count, dtype: int64

In [4]:
# Remove Intent type "None"
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.intent.value_counts()

Before:  88
After:  82


intent
Find Alternative    23
Filter Spam         20
Delete Account      17
Sync Accounts        9
Change Password      8
Export Data          5
Name: count, dtype: int64

In [5]:
# Generate cluster labels for short text dataset
dataset = df.data.values.tolist()

In [6]:
len(dataset)

82

In [7]:
%%time
model = ITER_DBSCAN(initial_distance=0.3, initial_minimum_samples=16, delta_distance=0.01, delta_minimum_samples=1, max_iteration=15)

CPU times: user 12 μs, sys: 1 μs, total: 13 μs
Wall time: 16.9 μs


In [8]:
%%time
labels = model.fit_predict(dataset)

Loading Tensorflow model....
Model Loaded.
CPU times: user 7.25 s, sys: 1.64 s, total: 8.9 s
Wall time: 9.4 s


In [9]:
df['cluster_ids'] = labels

In [10]:
df.cluster_ids.value_counts()

cluster_ids
-1    33
 0    13
 1    12
 2     5
 3     5
 4     4
 6     4
 5     3
 7     3
Name: count, dtype: int64

In [11]:
df.loc[df.cluster_ids == 0]

,data,intent,cluster_ids
1,How do I delete my Facebook account?,Delete Account,0
9,How can I delete my 160by2 account?,Delete Account,0
10,How can I permanently delete my Yahoo mail acc...,Delete Account,0
12,How to delete my imgur account?,Delete Account,0
14,How to delete a Sify Mail account,Delete Account,0
15,How to permanently delete a 37signals ID,Delete Account,0
16,How can I delete my Hunch account?,Delete Account,0
75,How can I delete my Twitter account?,Delete Account,0
76,How do I delete my LinkedIn account?,Delete Account,0
77,How do I delete my Gmail account?,Delete Account,0


In [12]:
evaluate_dataset = EvaluateDataset(filename=filepath, filetype='xlsx', text_column='data', 
                                   target_column='intent')

In [13]:
parameters = [
             {
               "distance":0.3, 
               "minimum_samples":16, 
               "delta_distance":0.01, 
               "delta_minimum_samples":1, 
               "max_iteration":15
             },
             {
               "distance":0.25, 
               "minimum_samples":14, 
               "delta_distance":0.01, 
               "delta_minimum_samples":1, 
               "max_iteration":12
             }, 
             {
               "distance":0.28, 
               "minimum_samples":12, 
               "delta_distance":0.01, 
               "delta_minimum_samples":1, 
               "max_iteration":12
             }
             ]

In [14]:
%%time
results = evaluate_dataset.evaulate_iter_dbscan(parameters)
result_df = pd.DataFrame.from_dict(results)

Loading Tensorflow model....
Model Loaded.


100%|██████████| 3/3 [00:00<00:00,  4.43it/s]

CPU times: user 7.82 s, sys: 2.17 s, total: 9.99 s
Wall time: 9.99 s


In [15]:
result_df

,distance,minimum_samples,delta_distance,delta_minimum_samples,max_iteration,time,percentage_labelled,clusters,noisy_clusters,homogeneity_score,completeness_score,normalized_mutual_info_score,adjusted_mutual_info_score,adjusted_rand_score,accuracy,precision,recall,f1,intents
0,0.30,16,0.01,1,15,0.11,56.82,8,0,0.76,0.88,0.81,0.79,0.81,0.852273,75.0,85.2,79.7,5
1,0.25,14,0.01,1,12,0.06,42.05,6,0,0.70,0.82,0.76,0.73,0.74,0.818182,72.4,81.8,76.6,5
2,0.28,12,0.01,1,12,0.06,46.59,7,0,0.73,0.85,0.79,0.77,0.78,0.840909,74.1,84.1,78.7,5
